# 04 — OE3: Cointegración y corrección de error

Contrasta **H5** (relación de largo plazo) y responde **PI3** (velocidad de ajuste).

Trabaja con las variables en **NIVEL (log)** que OE1 clasificó como **I(1)**:
valor de la cartera del sector, cobre, tipo de cambio, tasa.

Secuencia: ARDL bounds (principal) -> Johansen (robustez) -> VECM (alpha, beta).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import config as C
from src import cointegration as ci

## 1. Construir las series en nivel (log)
IMPORTANTE: aquí van NIVELES, no diferencias (a diferencia de OE2).

In [ ]:
g = pd.read_parquet(C.DATA_INTERIM / 'raw_macro_global.parquet')
px = pd.read_parquet(C.DATA_INTERIM / 'raw_precios.parquet')

# Valor de la cartera (ejemplo anillo 1: ANTO.L) y cobre, en log-nivel mensual
valor = np.log(px[['ANTO.L']].resample('ME').last())
cobre = np.log(g[['cobre']].resample('ME').last())
usdclp = np.log(pd.read_parquet(C.DATA_INTERIM / 'raw_macro_local_fred.parquet')[['usdclp']].resample('ME').last())

niveles = pd.concat([valor, cobre, usdclp], axis=1).loc['2004':'2024'].dropna()
niveles.columns = ['valor', 'cobre', 'usdclp']
niveles.tail()

## 2. ARDL bounds test (principal)
Si F > cota superior I(1) (tabla Pesaran-Shin-Smith 2001) => hay cointegración.

In [ ]:
out = ci.ardl_bounds(niveles['valor'], niveles[['cobre','usdclp']], maxlag=6)
print('AR lags:', out['ar_lags'], '| DL lags:', out['dl_lags'])
print('F-bounds stat:', round(out['F_stat'], 3))
print(out['bounds_test'])  # incluye valores críticos por nivel de significancia

In [ ]:
# Relación de largo plazo y término de corrección de error desde el UECM
print(out['uecm_res'].summary())

## 3. Johansen (robustez)

In [ ]:
tab_jo = ci.johansen(niveles, det_order=0, k_ar_diff=2)
display(tab_jo[['r<=','traza_stat','cv_95','coint_traza_95','maxeig_stat','cvm_95']].round(3))
tab_jo.to_csv(C.TABLES / 'oe3_johansen.csv', index=False)

## 4. VECM — vector de largo plazo (beta) y velocidad de ajuste (alpha)

In [ ]:
rango = ci.rango_cointegracion(niveles, det_order=0, k_ar_diff=2)
r = max(1, rango.rank)
vecm = ci.estimar_vecm(niveles, k_ar_diff=2, coint_rank=r)
print('Rango de cointegración:', r)
print('\nbeta (relación de largo plazo, normalizada en la 1a variable):')
print(pd.DataFrame(vecm.beta, index=niveles.columns).round(4))
print('\nalpha (velocidad de ajuste / corrección de error):')
print(pd.DataFrame(vecm.alpha, index=niveles.columns).round(4))

## 5. Lectura (OE3)
- ¿Rechaza el bounds test / Johansen la ausencia de cointegración? -> contrasta **H5**.
- Signo y magnitud de **beta**: ¿la relación de largo plazo cobre↑ -> valor↑ es coherente con H1?
- **alpha** de la ecuación del valor: debe ser **negativo y significativo** (corrige desviaciones).
  Su magnitud = velocidad de ajuste (responde PI3).

Próximo: `05_oe4_var_irf.ipynb` (si cointegran, VECM->IRF; si no, VAR en diferencias).